# Bài tập về nhà Buổi 2 - Xác suất thống kê cho AI
**Họ và tên:** Phan Gia Huy
**Dữ liệu sử dụng:** Iris Dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Style đồ thị
sns.set_theme(style="whitegrid")

# Tải Iris Dataset
df = sns.load_dataset("iris")

In [ ]:
# THỐNG KÊ MÔ TẢ & ĐẶC TRƯNG

# Hiển thị thông tin cơ bản
print("5 dòng đầu của dữ liệu:")
display(df.head())
print(f"\nSố dòng: {df.shape[0]}, Số cột: {df.shape[1]}")
print("\nKiểu dữ liệu của các cột:")
print(df.dtypes)

# Thống kê mô tả cho các biến số
numeric_cols = df.select_dtypes(include=[np.number])

stats_df = pd.DataFrame({
    'Mean': numeric_cols.mean(),
    'Median': numeric_cols.median(),
    'Mode': numeric_cols.mode().iloc[0], # Lấy mode đầu tiên nếu có nhiều mode
    'Var': numeric_cols.var(),
    'Std': numeric_cols.std(),
    'Min': numeric_cols.min(),
    'Max': numeric_cols.max(),
    'Q1': numeric_cols.quantile(0.25),
    'Q3': numeric_cols.quantile(0.75)
})
stats_df['IQR'] = stats_df['Q3'] - stats_df['Q1']
print("\nBảng thống kê mô tả chi tiết:")
display(stats_df)

# Thống kê mean, std theo từng loài
grouped_stats = df.groupby('species').agg(['mean', 'std'])
print("\nThống kê Mean và Std theo từng loài (species):")
display(grouped_stats)

**Nhận xét Phần 1:**
- Nhìn vào bảng thống kê theo nhóm, loài **setosa** có sự khác biệt rõ rệt nhất so với hai loài còn lại (versicolor và virginica), đặc biệt là ở hai biến `petal_length` (trung bình chỉ 1.46 so với 4.26 và 5.55) và `petal_width` (trung bình 0.24 so với 1.32 và 2.02).

In [ ]:
# PHÂN PHỐI XÁC SUẤT

# Vẽ Histogram + KDE cho từng biến số
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()
for i, col in enumerate(numeric_cols.columns):
    sns.histplot(df[col], kde=True, ax=axes[i], color='blue', stat='density')
    axes[i].set_title(f'Histogram & KDE của {col}')
plt.tight_layout()
plt.show()

# Vẽ Boxplot từng biến theo loài
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()
for i, col in enumerate(numeric_cols.columns):
    sns.boxplot(data=df, x='species', y=col, ax=axes[i], palette='Set2')
    axes[i].set_title(f'Boxplot của {col} theo Species')
plt.tight_layout()
plt.show()

# Mô phỏng phân phối Normal trên biến sepal_width
col_to_sim = 'sepal_width'
mu = df[col_to_sim].mean()
sigma = df[col_to_sim].std()

# Sinh 1000 mẫu từ phân phối chuẩn lý thuyết
simulated_data = np.random.normal(mu, sigma, 1000)

plt.figure(figsize=(8, 5))
# Vẽ histogram dữ liệu thật
sns.histplot(df[col_to_sim], kde=False, stat='density', label='Dữ liệu thực tế', color='skyblue')
# Vẽ KDE của dữ liệu mô phỏng
sns.kdeplot(simulated_data, color='red', label='PDF Lý thuyết (Normal)', linewidth=2)
plt.title(f"So sánh {col_to_sim} với Phân phối chuẩn")
plt.legend()
plt.show()

**Nhận xét Phần 2:**
- Hình dạng phân phối: `sepal_width` có dạng hình chuông, khá gần với phân phối chuẩn. Trong khi đó, `petal_length` và `petal_width` có dạng phân phối bimodal (2 đỉnh rõ rệt), nguyên nhân là do sự phân tách hình thái quá lớn của loài setosa so với phần còn lại.
- Mô phỏng PDF: Đường phân phối chuẩn lý thuyết (màu đỏ) khớp khá tốt với histogram thực tế của `sepal_width`, chứng tỏ biến này tuân theo phân phối chuẩn.

In [ ]:
# PHÂN TÍCH ĐA BIẾN & TƯƠNG QUAN

# Ma trận hiệp phương sai và Tương quan
cov_matrix = numeric_cols.cov()
cor_matrix = numeric_cols.corr()

print("Ma trận hiệp phương sai (Covariance):")
display(cov_matrix)

# Vẽ Heatmap tương quan
plt.figure(figsize=(8, 6))
sns.heatmap(cor_matrix, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)
plt.title("Heatmap Ma trận Tương quan")
plt.show()

# Vẽ Pairplot tô màu theo loài
sns.pairplot(df, hue='species', palette='Set1', diag_kind='kde', markers=["o", "s", "D"])
plt.suptitle("Pairplot các biến theo Species", y=1.02)
plt.show()

**Nhận xét Phần 3:**
- Cặp biến tương quan mạnh nhất là `petal_length` và `petal_width` với hệ số $r = 0.96$ (tương quan thuận cực kỳ mạnh). Điều này cho thấy rõ dấu hiệu của hiện tượng **đa cộng tuyến** nếu đưa cả hai biến này vào cùng một mô hình tuyến tính.
- Qua Pairplot, loài setosa tách biệt hoàn toàn so với hai loài kia trên hầu hết các trục tọa độ (nhất là trục petal). Versicolor và virginica có sự chồng lấn nhẹ nhưng vẫn có xu hướng phân tách tuyến tính.

In [ ]:
# XÁC SUẤT & ĐỊNH LÝ BAYES + LỌC SPAM

# Tính xác suất hậu nghiệm P(B|+)
P_B = 0.01          # Tỉ lệ mắc bệnh
P_not_B = 1 - P_B   # Tỉ lệ không mắc bệnh
P_pos_givenB = 0.99 # Độ nhạy P(+|B)
P_pos_givnNB = 0.05 # Dương tính giả P(+|~B)

# Xác suất toàn phần P(+)
P_pos = (P_pos_givenB * P_B) + (P_pos_givnNB * P_not_B)

# Hậu nghiệm P(B|+)
P_B_given_pos = (P_pos_givenB * P_B) / P_pos

print(f"Xác suất thực sự mắc bệnh khi test dương tính P(B|+) = {P_B_given_pos:.4f} (~{P_B_given_pos*100:.2f}%)")

# Khảo sát P(B|+) khi P(B) thay đổi từ 0.001 đến 0.2
P_B_range = np.linspace(0.001, 0.2, 100)
P_not_B_range = 1 - P_B_range
P_pos_range = (P_pos_givenB * P_B_range) + (P_pos_givnNB * P_not_B_range)
P_B_given_pos_range = (P_pos_givenB * P_B_range) / P_pos_range

plt.figure(figsize=(8, 5))
plt.plot(P_B_range, P_B_given_pos_range, color='purple', linewidth=2)
plt.xlabel("Tỉ lệ mắc bệnh trong dân số P(B)")
plt.ylabel("Xác suất hậu nghiệm P(B|+)")
plt.title("Sự thay đổi của P(B|+) theo tỉ lệ mắc bệnh")
plt.grid(True)
plt.show()


# BỘ LỌC SPAM NAIVE BAYES

print("\n--- KẾT QUẢ BONUS: SPAM FILTER NAIVE BAYES ---")
# Giả sử tập dữ liệu huấn luyện có 50% Spam, 50% Ham
p_spam = 0.5
p_ham = 0.5

# Xác suất xuất hiện từ khóa với điều kiện (Spam/Ham)
# P(word | Class)
word_probs = {
    'free': {'spam': 0.8, 'ham': 0.1},
    'money': {'spam': 0.7, 'ham': 0.2},
    'meeting': {'spam': 0.05, 'ham': 0.6}
}

def predict_spam(email_text):
    words = email_text.lower().split()
    # Khởi tạo điểm số theo Naive Bayes (bỏ qua mẫu số chung P(email))
    score_spam = p_spam
    score_ham = p_ham
    
    for w in words:
        if w in word_probs:
            score_spam *= word_probs[w]['spam']
            score_ham *= word_probs[w]['ham']
            
    # Chuẩn hóa lại thành xác suất
    p_spam_given_email = score_spam / (score_spam + score_ham)
    return p_spam_given_email

emails = [
    "free money",
    "meeting schedule",
    "free meeting"
]

for email in emails:
    prob = predict_spam(email)
    label = "SPAM" if prob > 0.5 else "HAM"
    print(f"Email: '{email:<18}' => Phân loại: {label} (P(Spam) = {prob:.4f})")

**Nhận xét Phần 4:**
- Mặc dù test có độ nhạy rất cao (99%), nhưng vì tỉ lệ người mắc bệnh trong dân số quá thấp ($P(B) = 0.01$), lượng "dương tính giả" từ tập người không bệnh (99% dân số x 5% = 4.95%) áp đảo hoàn toàn lượng "dương tính thật" (1% dân số x 99% = 0.99%). Do đó, kết quả trông có vẻ "phản trực giác" vì thực tế khi dương tính, xác suất mắc bệnh thật sự chỉ khoảng 16.67%.
- Nhìn vào đồ thị khảo sát, nếu tỉ lệ mắc bệnh gốc $P(B)$ càng tăng lên, thì độ tin cậy của xét nghiệm dương tính $P(B|+)$ mới càng dốc lên và tiệm cận 1.